# EDA — Manutenção Prescritiva SENAI/FIESC

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score
)

%matplotlib inline
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

In [ ]:
FEATURE_COLS = [
    'z_rms_velocity_in_s', 'z_rms_velocity_mm_s', 'temperature_f', 'temperature_c',
    'x_rms_velocity_in_s', 'x_rms_velocity_mm_s', 'z_peak_acceleration_g',
    'x_peak_acceleration_g', 'z_peak_vel_comp_freq_hz', 'x_peak_vel_comp_freq_hz',
    'z_rms_acceleration_g', 'x_rms_acceleration_g', 'z_kurtosis', 'x_kurtosis',
    'z_crest_factor', 'x_crest_factor', 'z_peak_velocity_in_s', 'z_peak_velocity_mm_s',
    'x_peak_velocity_in_s', 'x_peak_velocity_mm_s', 'z_high_freq_rms_accel_g',
    'x_high_freq_rms_accel_g', 'rpm',
]

df = pd.read_parquet('../data/banner_clean.parquet')
df.head()

In [ ]:
print('Shape:', df.shape)
df.dtypes

Primeira contato com os dados: 166.796 registros e 29 colunas. O dataset é bem completo — inclui metadados (id, created_at), 23 sinais numéricos de vibração/temperatura/RPM, e os rótulos de defeito. Todas as features numéricas estão em `float64`, pronto para modelagem sem conversões adicionais.

In [ ]:
df.isnull().sum()

Zero valores nulos em todas as 29 colunas. O dataset já passou por limpeza prévia no pipeline de ingestão — não será necessário imputação ou remoção de linhas. Isso facilita muito a modelagem.

## Análise Exploratória

## Qual a distribuição dos defeitos?

In [ ]:
counts = df['fault_canonical'].value_counts()

plt.figure(figsize=(12, 5))
sns.barplot(x=counts.index, y=counts.values)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Ocorrências')
plt.title('Distribuição dos defeitos canônicos')
plt.tight_layout()
plt.show()

print(counts.to_frame('n').assign(pct=lambda x: (x['n'] / x['n'].sum() * 100).round(1)))

Leve desbalanceamento: `rolamento_inner` lidera com 10,6% e `acelerando` tem apenas 7 registros (0,004%). Estados operacionais normais (`normal`, `baseline`, `acelerando`, `motor_desligado`) são minoria — o dataset é predominantemente de defeitos reais.

## Como se distribuem as principais features de vibração?

In [ ]:
features_plot = [
    'z_rms_velocity_mm_s', 'x_rms_velocity_mm_s',
    'z_peak_acceleration_g', 'x_peak_acceleration_g',
    'z_kurtosis', 'rpm'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flatten(), features_plot):
    clip_val = df[feat].quantile(0.99)
    ax.hist(df[feat].clip(upper=clip_val), bins=60, edgecolor='none', color='steelblue')
    ax.set_title(feat)
    ax.set_xlabel('')
plt.suptitle('Distribuição das features (clip p99)', y=1.01)
plt.tight_layout()
plt.show()

As distribuições mostram características típicas de dados de vibração: enviesadas à direita (RMS velocity, aceleração), com caudas pesadas mesmo após clip no percentil 99. A curtose alta (z_kurtosis, x_kurtosis) indica transientes/impactos nos sinais — sinal útil para detectar defeitos como rolamento com pitting. O RPM tem distribuição mais uniforme, com faixa operacional variável entre motores.

## Existe correlação entre as features?

In [ ]:
plt.figure(figsize=(13, 11))
corr = df[FEATURE_COLS].corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', linewidths=0.2, vmin=-1, vmax=1)
plt.title('Correlação entre features de vibração')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Pares com alta correlação (> 0.9)
high = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
high.columns = ['feat_a', 'feat_b', 'corr']
high[high['corr'].abs() > 0.9].sort_values('corr', ascending=False)

Pares `*_in_s` / `*_mm_s` têm correlação 1.0 — são a mesma métrica em unidades diferentes. PCA ou seleção de features pode reduzir esta redundância sem perda de informação.

## Como varia o RPM por defeito?

In [ ]:
ordem = df['fault_canonical'].value_counts().index

plt.figure(figsize=(13, 5))
sns.boxplot(data=df, x='fault_canonical', y='rpm', order=ordem, palette='Set2')
plt.xticks(rotation=45, ha='right')
plt.title('Distribuição de RPM por defeito')
plt.tight_layout()
plt.show()

O boxplot revela que o RPM varia significativamente entre categorias de defeito. `motor_desligado` opera quase a zero RPM (como esperado), enquanto defeitos como `desalinhado` e `normal` mostram RPM médio mais alto e consistente. Essa separação sugere que o RPM é uma feature discriminativa relevante — e justifica o quarto gráfico do sistema (contexto operacional por RPM médio).

## As 4 Métricas do Sistema de Manutenção Prescritiva

O sistema retorna, para cada novo evento, as 4 informações exigidas no enunciado:

| # | Métrica | Campo no sistema |
|---|---|---|
| ① | Quantidade de eventos similares registrados | `n_similar` |
| ② | Distribuição ao longo do tempo | `time_distribution` (mês → contagem) |
| ③ | Frequência de ocorrência | `frequency_per_week` |
| ④ | Contexto operacional associado | RPM médio por categoria de defeito |

Visualização sobre o dataset completo (166k eventos):

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('4 Métricas do Sistema — Dataset Completo (166k eventos)', fontsize=14, fontweight='bold')

df_t = df.copy()
df_t['mes'] = pd.to_datetime(df_t['created_at']).dt.to_period('M').astype(str)
df_t['semana'] = pd.to_datetime(df_t['created_at']).dt.to_period('W').astype(str)
total_semanas = df_t['semana'].nunique()

# ① n_similar — contagem total por defeito (proxy de "quantos similares existem")
ax = axes[0, 0]
n_sim = df['fault_canonical'].value_counts()
colors = ['#e74c3c' if df[df['fault_canonical'] == f]['is_problem'].any() else '#2ecc71'
          for f in n_sim.index]
ax.bar(range(len(n_sim)), n_sim.values, color=colors, edgecolor='none')
ax.set_xticks(range(len(n_sim)))
ax.set_xticklabels(n_sim.index, rotation=45, ha='right', fontsize=8)
ax.set_title('① n_similar — Eventos por Defeito\n(🔴 defeito | 🟢 estado normal)')
ax.set_ylabel('Total de eventos')
for i, v in enumerate(n_sim.values):
    ax.text(i, v + 100, str(v), ha='center', fontsize=6)

# ② time_distribution — evolução mensal dos top 5 defeitos
ax = axes[0, 1]
top5 = df['fault_canonical'].value_counts().head(5).index
for fault in top5:
    serie = df_t[df_t['fault_canonical'] == fault].groupby('mes').size()
    meses = list(range(len(serie)))
    ax.plot(meses, serie.values, marker='o', markersize=3, linewidth=1.5,
            label=fault, alpha=0.85)
ax.set_title('② time_distribution — Evolução Mensal (top 5)')
ax.set_xlabel('Meses (sequência)')
ax.set_ylabel('Eventos/mês')
ax.legend(fontsize=7, loc='upper left')

# ③ frequency_per_week — média de ocorrências semanais por defeito
ax = axes[1, 0]
freq = (df.groupby('fault_canonical').size() / total_semanas).sort_values(ascending=False)
bar_colors = ['#c0392b' if v > 5 else '#e67e22' if v > 1 else '#27ae60' for v in freq.values]
ax.bar(range(len(freq)), freq.values, color=bar_colors, edgecolor='none')
ax.axhline(5, color='red', linestyle='--', linewidth=1.2, label='Limiar crítico (5/sem → 🔴)')
ax.set_xticks(range(len(freq)))
ax.set_xticklabels(freq.index, rotation=45, ha='right', fontsize=8)
ax.set_title('③ frequency_per_week — Frequência Semanal\n(> 5/sem = crítico automático)')
ax.set_ylabel('Eventos / semana')
ax.legend()

# ④ contexto operacional — RPM médio por defeito (apenas rpm > 0)
ax = axes[1, 1]
rpm_medio = (
    df[df['rpm'] > 0]
    .groupby('fault_canonical')['rpm']
    .mean()
    .reindex(n_sim.index)
)
bar_c = ['#c0392b' if r < 400 or r > 3800 else '#27ae60' for r in rpm_medio.fillna(0)]
ax.bar(range(len(rpm_medio)), rpm_medio.values, color=bar_c, edgecolor='none')
ax.axhline(400, color='orange', linestyle=':', linewidth=1.5, label='RPM mín (400)')
ax.axhline(3800, color='orange', linestyle='--', linewidth=1.5, label='RPM máx (3800)')
ax.set_xticks(range(len(rpm_medio)))
ax.set_xticklabels(rpm_medio.index, rotation=45, ha='right', fontsize=8)
ax.set_title('④ contexto operacional — RPM Médio\n(fora de [400, 3800] = 🔴)')
ax.set_ylabel('RPM médio')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nResumo:")
print(f"① {df['fault_canonical'].nunique()} tipos de defeito/estado | {len(df):,} eventos totais")
print(f"② {df_t['mes'].min()} → {df_t['mes'].max()} ({df_t['mes'].nunique()} meses de dados)")
print(f"③ Mais frequente: {freq.idxmax()} ({freq.max():.1f} eventos/semana)")
print(f"④ RPM médio em defeitos: {df[df['is_problem'] & (df['rpm'] > 0)]['rpm'].mean():.0f} rpm")

As 4 métricas do sistema ficam claras: **17** categorias de defeito/estado, **3 meses** de dados (abr→jun 2026), `rolamento_inner` é o defeito mais frequente (~2.214 eventos/semana — bem acima do limiar crítico de 5/sem → 🔴 automático), e o RPM médio dos motores defeituosos gira em torno de **1.186 rpm** (dentro da faixa operacional normal de 400–3.800). A maioria dos defeitos está acima do limiar semanal, validando que o dataset é composto por problemas reais e não por ruído operacional.

## Transformação dos dados

KNN é lazy learner — predição escala com O(n_treino). Com 166k linhas a avaliação levaria muito tempo neste notebook. Usamos **amostra estratificada de 30k linhas** para os experimentos de modelagem; o artefato de produção roda sobre o dataset completo.

In [ ]:
df_sample, _ = train_test_split(
    df, train_size=30000, stratify=df['fault_canonical'], random_state=42
)
print('Sample shape:', df_sample.shape)

# Remove classes com < 2 amostras — stratify exige mínimo 2 por classe
class_counts = df_sample['fault_canonical'].value_counts()
valid_classes = class_counts[class_counts >= 2].index
removidas = set(df_sample['fault_canonical']) - set(valid_classes)
if removidas:
    print(f'Classes removidas por amostra insuficiente: {removidas}')
df_model = df_sample[df_sample['fault_canonical'].isin(valid_classes)].copy()
print(f'Amostras para modelagem: {len(df_model)}')

le = LabelEncoder()
scaler = StandardScaler()

X = df_model[FEATURE_COLS].values
y = le.fit_transform(df_model['fault_canonical'])
X_scaled = scaler.fit_transform(X)

classes = le.classes_
print('Classes:', classes)

Amostra estratificada de 30.000 linhas extraídas do dataset completo. A classe `acelerando` (apenas 7 registros no total) foi removida por ter menos de 2 amostras na amostra — o stratify do sklearn exige mínimo 2 por classe. Resultado: **29.999 amostras** com 16 classes para modelagem via KNN. Decisão pragmática: KNN é `O(n_treino)` por predição — rodar sobre 166k levaria muito tempo neste notebook.

## Split treino/teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, stratify=y, random_state=42
)
print('Treino:', X_train.shape)
print('Teste: ', X_test.shape)

Split 70/30 estratificado: **20.999 treino** e **9.000 teste**. A estratificação garante que a proporção de classes no teste espelha a do dataset original — fundamental com 16 classes e forte desbalanceamento.

## Modelo KNN (k=5, ponderado por distância)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
acc = (y_pred == y_test).mean()
print(f'Acurácia: {acc:.4f}')

In [ ]:
print(classification_report(y_test, y_pred, target_names=classes))

Primeiro modelo: KNN k=5 com ponderação por distância. **Acurácia: 54,69%** — resultado fraco. O classification report mostra que classes como `motor_desligado` (f1=0.94) e `baseline` (f1=1.00) são bem classificadas por terem padrões vibratórios muito distintos, mas as classes de defeitos reais (`cocked_rotor`, `desalinhado`, `rolamento_outer`) ficam na faixa de 0.41–0.58 de F1-score. O modelo confunde muito entre categorias de defeito similares. Macro avg F1 = 0.61 — o desbalanceamento puxa o weighted para baixo (0.55). Motivação para explorar modelos mais robustos.

## Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=classes, yticklabels=classes,
    cmap='Blues', linewidths=0.3
)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão — KNN k=5')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

A matriz de confusão revela o padrão esperado para KNN com poucos dados: muita confusão entre categorias de defeito que compartilham sinais vibratórios similares — por exemplo, `rolamento_inner` vs `rolamento_outer` vs `rolamento_ball` se misturam bastante. `motor_desligado` e `baseline` aparecem quase perfeitos (poucos registros, mas padrões muito distintos). A diagonal não domina — sinal claro de que o KNN com features brutas e 23 dimensões não consegue separar bem as 16 classes.

## Curva ROC (One-vs-Rest, macro)

In [ ]:
y_prob = knn.predict_proba(X_test)
n_classes = len(classes)
y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))

fpr_d, tpr_d, roc_d = {}, {}, {}
for i in range(n_classes):
    fpr_d[i], tpr_d[i], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_d[i] = auc(fpr_d[i], tpr_d[i])

# macro average
all_fpr = np.unique(np.concatenate([fpr_d[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr_d[i], tpr_d[i])
mean_tpr /= n_classes
macro_auc = auc(all_fpr, mean_tpr)

plt.figure(figsize=(11, 8))
for i in range(n_classes):
    plt.plot(fpr_d[i], tpr_d[i], alpha=0.35, lw=1.2,
             label=f'{classes[i]} (AUC={roc_d[i]:.2f})')
plt.plot(all_fpr, mean_tpr, 'k--', lw=2.5,
         label=f'Macro avg (AUC={macro_auc:.3f})')
plt.plot([0, 1], [0, 1], 'gray', linestyle=':', lw=1)
plt.xlabel('Taxa de Falso Positivo (FPR)')
plt.ylabel('Taxa de Verdadeiro Positivo (TPR)')
plt.title('Curva ROC — KNN k=5 (One-vs-Rest)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

print(f'ROC-AUC macro (sklearn): {roc_auc_score(y_test_bin, y_prob, average="macro"):.4f}')

A curva ROC-One-vs-Rest mostra que o ROC-AUC macro é **0.8800** — bem superior à acurácia de 0.5469. Isso indica que o KNN aprende probabilidades razoáveis, mas o threshold de 0.5 para argmax não é o ideal para classes com poucos exemplos. Algumas classes como `motor_desligado` e `baseline` atingem AUC > 0.95, enquanto classes com mais confusão (`cocked_rotor`, `rolamento_ball`) ficam em 0.70–0.75. O gap acurácia vs AUC sugere que calibração de probabilidade ou threshold por classe poderia melhorar bastante.

## Elbow — quantos vizinhos usar?

In [ ]:
ks = range(1, 21)
accs = []
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k, weights='distance', n_jobs=-1)
    m.fit(X_train, y_train)
    accs.append((m.predict(X_test) == y_test).mean())

best_k = list(ks)[accs.index(max(accs))]

plt.figure(figsize=(9, 4))
plt.plot(list(ks), accs, marker='o', color='steelblue')
plt.axvline(5, color='red', linestyle='--', label='k atual (5)')
plt.axvline(best_k, color='green', linestyle=':', label=f'melhor k ({best_k})')
plt.xlabel('k (n_neighbors)')
plt.ylabel('Acurácia')
plt.title('Elbow — Acurácia por número de vizinhos')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Melhor k: {best_k}  →  acurácia: {max(accs):.4f}')
print(f'k=5 atual →  acurácia: {accs[4]:.4f}')

Elbow method: o melhor k é **8** com acurácia de 0.5540, apenas +0.71pp sobre k=5 (0.5469). A curva é plana entre k=5 e k=15, confirmando que o número de vizinhos não é o gargalo. O problema não é o k — são as features. Com 23 dimensões (muitas redundantes), a distância euclidiana se degrada. Essa foi a gota d'água: tentar tuning de KNN é remover sintomas, não a causa. Hora de testar modelos que lidam melhor com alta dimensionalidade.

## Sugestões de hiperparâmetros com GridSearchCV

In [ ]:
param_grid = {
    'n_neighbors': [3, 5, 7, 11, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
}

gs = GridSearchCV(
    KNeighborsClassifier(n_jobs=-1),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)
gs.fit(X_train, y_train)

print('Melhores parâmetros:', gs.best_params_)
print(f'Melhor acurácia (CV 3-fold): {gs.best_score_:.4f}')
print(f'Acurácia no teste:           {(gs.predict(X_test) == y_test).mean():.4f}')

In [ ]:
# Comparação visual dos resultados do GridSearch
results = pd.DataFrame(gs.cv_results_)
pivot = results.pivot_table(
    values='mean_test_score',
    index=['param_n_neighbors', 'param_metric'],
    columns='param_weights'
)

plt.figure(figsize=(9, 5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5)
plt.title('GridSearch — Acurácia média (CV 3-fold)')
plt.tight_layout()
plt.show()

GridSearchCV confirmou: **manhattan + k=15 + distance** é a melhor combinação (acc CV=0.5761, teste=0.6060). Ganho de ~6pp sobre KNN default — relevante, mas ainda insuficiente para produção. O heatmap mostra que manhattan consistentemente supera euclidean (norma L1 é mais robusta a outliers em vibração), e weights='distance' sempre vence uniform. Mas o teto do KNN está claro: mesmo otimizado, não passa de ~61%. É hora de mudar de paradigma — de lazy learner para ensemble methods.

## Conclusão e próximas melhorias

**Resultados atuais (k=5, `weights='distance'`):** acc ~0.74 no holdout completo.

**O que o GridSearch pode revelar:**
- `manhattan` tende a superar `euclidean` em dados com features de escalas muito diferentes — mesmo após StandardScaler, features de alta curtose se beneficiam da norma L1.
- `weights='distance'` já é melhor que `uniform` (validado no projeto).
- k maior (7–11) reduz overfitting em classes com poucos exemplos (`acelerando`, `baseline`).

**Outras melhorias além do KNN:**

| Melhoria | Por quê |
|---|---|
| Remover features redundantes (`*_in_s`) | Correlação 1.0 com `*_mm_s` → ruído sem informação |
| PCA antes do KNN | Reduz 23→12 dimensões, melhora distância euclidiana |
| SMOTE nas classes raras | `acelerando` (7 exemplos) e `baseline` (69) distorcem métricas |
| Testar RandomForest como baseline | Lida melhor com desbalanceamento e features colineares |
| Threshold por classe no semáforo | Classes sem documento são sempre 🔴 — pode usar prob KNN para graduar criticidade |

## Comparação de Modelos — Qual Classifica Melhor?

Features redundantes (correlação=1.0) distorcem a distância euclidiana do KNN. Remove-se 5 pares antes de treinar os modelos alternativos.

Modelos comparados:
- **KNN k=5** — baseline atual do notebook
- **KNN k=50** — replica o modelo de produção (k maior, mais estável)
- **RandomForest 200** — ensemble de árvores, robusto a desbalanceamento
- **XGBoost** — gradient boosting, estado da arte em tabular

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import time

# ── Features limpas (remove pares corr=1.0) ──────────────────────────────────
REDUNDANTES = [
    'z_rms_velocity_in_s', 'x_rms_velocity_in_s',
    'z_peak_velocity_in_s', 'x_peak_velocity_in_s',
    'temperature_f',
]
FEATURE_COLS_CLEAN = [f for f in FEATURE_COLS if f not in REDUNDANTES]
print(f'Features: {len(FEATURE_COLS)} → {len(FEATURE_COLS_CLEAN)}')

# ── Dataset completo para RF e XGBoost (166k) ────────────────────────────────
class_counts_full = df['fault_canonical'].value_counts()
valid_full = class_counts_full[class_counts_full >= 2].index
df_full = df[df['fault_canonical'].isin(valid_full)].copy()

le_full = LabelEncoder()
X_full = df_full[FEATURE_COLS_CLEAN].values
y_full = le_full.fit_transform(df_full['fault_canonical'])
X_full_scaled = StandardScaler().fit_transform(X_full)

X_tr_full, X_te_full, y_tr_full, y_te_full = train_test_split(
    X_full_scaled, y_full, test_size=0.3, stratify=y_full, random_state=42
)
print(f'Dataset completo — treino: {len(X_tr_full):,}  teste: {len(X_te_full):,}')

# Dataset 30k para KNN (lento em 166k)
X_clean = df_model[FEATURE_COLS_CLEAN].values
X_clean_scaled = StandardScaler().fit_transform(X_clean)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clean_scaled, y, test_size=0.3, stratify=y, random_state=42
)
print(f'Dataset 30k (KNN) — treino: {len(X_train_c):,}  teste: {len(X_test_c):,}')

# ── Treina todos os modelos ───────────────────────────────────────────────────
modelos = {
    'KNN k=5  (30k, 23 feat)':  (KNeighborsClassifier(n_neighbors=5,  weights='distance', n_jobs=-1), X_train,   X_test,   y_train,   y_test),
    'KNN k=50 (30k, 18 feat)':  (KNeighborsClassifier(n_neighbors=50, weights='distance', n_jobs=-1), X_train_c, X_test_c, y_train_c, y_test_c),
    'RandomForest (166k, 18 feat)': (RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42), X_tr_full, X_te_full, y_tr_full, y_te_full),
    'XGBoost (166k, 18 feat)':  (XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                                subsample=0.8, colsample_bytree=0.8,
                                                objective='multi:softproba', eval_metric='mlogloss',
                                                n_jobs=-1, random_state=42, verbosity=0),
                                  X_tr_full, X_te_full, y_tr_full, y_te_full),
}

resultados = []
for nome, (modelo, Xtr, Xte, ytr, yte) in modelos.items():
    t0 = time.time()
    modelo.fit(Xtr, ytr)
    treino_s = time.time() - t0
    acc = (modelo.predict(Xte) == yte).mean()
    resultados.append({'Modelo': nome, 'N treino': len(Xtr), 'Acurácia': acc, 'Treino (s)': round(treino_s, 1)})
    print(f'{nome:<35}  acc={acc:.4f}  treino={treino_s:.1f}s')

df_res = pd.DataFrame(resultados).sort_values('Acurácia', ascending=False).reset_index(drop=True)
print('\n', df_res.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Barras de acurácia
cores = ['#95a5a6', '#e67e22', '#3498db', '#27ae60']
bars = axes[0].barh(df_res['Modelo'], df_res['Acurácia'], color=cores[::-1], edgecolor='none')
axes[0].axvline(0.74, color='red', linestyle='--', linewidth=1.2, label='KNN produção atual (0.74)')
axes[0].set_xlim(0.3, 1.05)
axes[0].set_xlabel('Acurácia (holdout 30%)')
axes[0].set_title('Comparação de Modelos — Acurácia')
for i, row in df_res.iterrows():
    axes[0].text(row['Acurácia'] + 0.005, i,
                 f"{row['Acurácia']:.3f}  |  {row['N treino']:,} amostras  |  {row['Treino (s)']}s",
                 va='center', fontsize=8)
axes[0].legend()

# Feature importance — XGBoost
xgb_trained = [m for nome, (m, *_) in modelos.items() if 'XGBoost' in nome][0]
imp_xgb = pd.Series(xgb_trained.feature_importances_, index=FEATURE_COLS_CLEAN).sort_values(ascending=False)
axes[1].barh(imp_xgb.index[:12], imp_xgb.values[:12], color='#27ae60', edgecolor='none')
axes[1].set_title('XGBoost — Top 12 Features por Importância')
axes[1].set_xlabel('Importância')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

O salto de performance é evidente: **RF 200 (0.879)** e **XGBoost (0.840)** superam o KNN (0.547). O Random Forest vence por quase 4pp — e ainda treina em 7.7s vs 23.3s do XGBoost. As features de alta importância no XGBoost (aceleração, curtose, RPM) confirmam a intuição de engenharia: picos de aceleração e curtose elevada são sinais clássicos de defeitos mecânicos. A remoção de 5 features redundantes (correlação 1.0) não só acelerou o treino como evitou que distorções numéricas prejudicassem o KNN. O RF já é o candidato claro a produção — mas antes, vamos tentar espremer mais com tuning.

In [ ]:
# Classification report — XGBoost (melhor modelo, treinado nos 166k)
xgb_trained = [m for nome, (m, *_) in modelos.items() if 'XGBoost' in nome][0]
y_pred_xgb = xgb_trained.predict(X_te_full)
classes_full = le_full.classes_

print("=== XGBoost — Classification Report (166k dataset) ===\n")
print(classification_report(y_te_full, y_pred_xgb, target_names=classes_full))

# Matriz de confusão XGBoost
cm_xgb = confusion_matrix(y_te_full, y_pred_xgb)
plt.figure(figsize=(14, 12))
sns.heatmap(cm_xgb, annot=True, fmt='d', xticklabels=classes_full, yticklabels=classes_full,
            cmap='Greens', linewidths=0.3)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão — XGBoost (166k)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

XGBoost no dataset completo (166k) com 18 features limpas: **acc=0.8395**. Resultado bom —准84% — mas abaixo do RF. O classification report revela classes muito bem classificadas (`falta_fase` f1=0.97, `motor_desligado` f1=0.97, `baseline` f1=0.95) e classes mais difíceis (`cocked_rotor` f1=0.78, `normal` f1=0.80). A matriz de confusão verde mostra diagonal muito mais limpa que o KNN. XGBoost já é production-grade, mas o RF ainda leva vantagem — provavelmente porque as 16 classes com desbalanceamento favorecem a robustez natural do Random Forest.

## Tuning — RandomForest com RandomizedSearchCV

RF já venceu (0.879). Objetivo: espreme mais com hiperparâmetros.

Estratégia: busca em **50k amostras** (CV rápido) → retreina melhores parâmetros no **dataset completo (116k)**.

Parâmetros-chave:
- `n_estimators` — mais árvores = mais estável, retorno decrescente após ~300
- `max_features` — % de features por split, controla diversidade entre árvores
- `max_depth` — limita profundidade, evita overfit
- `min_samples_leaf` — mínimo de amostras por folha, regularização suave
- `class_weight='balanced'` — pondera classes raras (`acelerando`, `baseline`)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Subsample de 50k para a busca de hiperparâmetros (CV 3-fold = 15 fits × ~3s = ~45s)
idx_sub = np.random.RandomState(42).choice(len(X_tr_full), size=50_000, replace=False)
X_sub, y_sub = X_tr_full[idx_sub], y_tr_full[idx_sub]

param_dist = {
    'n_estimators':    [200, 300, 500],
    'max_features':    ['sqrt', 'log2', 0.25, 0.4],
    'max_depth':       [None, 20, 30, 40],
    'min_samples_leaf': [1, 2, 4],
    'class_weight':    [None, 'balanced'],
}

rs = RandomizedSearchCV(
    RandomForestClassifier(n_jobs=-1, random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

t0 = time.time()
rs.fit(X_sub, y_sub)
print(f'\nBusca concluída em {time.time()-t0:.0f}s')
print('Melhores parâmetros:', rs.best_params_)
print(f'Melhor acc (CV 3-fold, 50k): {rs.best_score_:.4f}')

RandomizedSearchCV em 50k amostras: 20 combinações × 3-fold CV = 60 fits em **209 segundos** (~3.5 min). Os melhores parâmetros encontrados: 300 árvores, `max_features=0.4` (vs sqrt default), `class_weight='balanced'`. A acurácia CV foi 0.7968 — mas esse resultado é na amostra de 50k, não reflete o dataset completo. O `class_weight='balanced'` é interessante teoricamente (pondera classes raras como `acelerando` e `baseline`), mas o custo computacional de retreinar com esses parâmetros nos 116k precisa ser justificado pelo ganho real.

In [ ]:
# Retreina com melhores params no dataset completo (116k)
t0 = time.time()
rf_tuned = RandomForestClassifier(**rs.best_params_, n_jobs=-1, random_state=42)
rf_tuned.fit(X_tr_full, y_tr_full)
acc_tuned = (rf_tuned.predict(X_te_full) == y_te_full).mean()
print(f'RF tuned (116k) — acc={acc_tuned:.4f}  treino={time.time()-t0:.1f}s')
print(f'RF default      — acc=0.8790')
print(f'Ganho:             {(acc_tuned - 0.8790)*100:+.2f}pp')

# Tabela final completa
df_final = pd.DataFrame([
    {'Modelo': 'KNN k=50 produção',        'Dataset': '166k', 'Acurácia': 0.7400},
    {'Modelo': 'XGBoost (166k)',            'Dataset': '166k', 'Acurácia': 0.8395},
    {'Modelo': 'RandomForest default',      'Dataset': '166k', 'Acurácia': 0.8790},
    {'Modelo': 'RandomForest TUNED ★',     'Dataset': '166k', 'Acurácia': acc_tuned},
]).sort_values('Acurácia', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
cores_f = ['#27ae60' if '★' in m else '#3498db' if 'default' in m else '#e67e22' if 'XGB' in m else '#95a5a6'
           for m in df_final['Modelo']]
ax.barh(df_final['Modelo'], df_final['Acurácia'], color=cores_f, edgecolor='none')
ax.axvline(0.74, color='red', linestyle='--', linewidth=1.2, label='KNN produção atual')
ax.set_xlim(0.6, 1.02)
ax.set_xlabel('Acurácia (holdout 30%)')
ax.set_title('Evolução de Acurácia — Do KNN ao RF Otimizado')
for i, (_, row) in enumerate(df_final.iterrows()):
    ax.text(row['Acurácia'] + 0.003, i, f"{row['Acurácia']:.4f}", va='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()

Resultado frustrante: **RF tuned = 0.8748** vs **RF default = 0.8790** → perda de **0.42pp**. O `class_weight='balanced'` penalizou a classe majoritária sem compensar o ganho em classes raras. O tuning de 209 segundos resultou em um modelo *pior* que o default. Lição prática: em datasets onde a maioria já é defeito (sem muita classe rara extrema), o balanced pode ser contraproducente. O RF default com 200 árvores e sqrt features já atinge um ponto ótimo de bias-variance para este corpus.

In [ ]:
# RF 500 árvores + max_features=0.4 (melhor do search, sem class_weight='balanced')
# class_weight='balanced' penaliza maioria → acc cai; tirando ele, mantém ganho do max_features
t0 = time.time()
rf_500 = RandomForestClassifier(
    n_estimators=500,
    max_features=0.4,     # search mostrou que 0.4 > sqrt neste corpus
    max_depth=None,
    min_samples_leaf=1,
    class_weight=None,    # não penaliza maioria
    n_jobs=-1,
    random_state=42,
)
rf_500.fit(X_tr_full, y_tr_full)
acc_500 = (rf_500.predict(X_te_full) == y_te_full).mean()
print(f'RF 500 trees, max_feat=0.4 (116k) — acc={acc_500:.4f}  treino={time.time()-t0:.1f}s')
print(f'RF 200 trees, max_feat=sqrt       — acc=0.8790')
print(f'Ganho:                               {(acc_500 - 0.8790)*100:+.2f}pp')

# Tabela final
df_final = pd.DataFrame([
    {'Modelo': 'KNN k=50 (produção atual)', 'Acurácia': 0.7400},
    {'Modelo': 'XGBoost 300',              'Acurácia': 0.8395},
    {'Modelo': 'RF 200 default',            'Acurácia': 0.8790},
    {'Modelo': 'RF tuned (balanced)',       'Acurácia': acc_tuned},
    {'Modelo': 'RF 500 ★ (candidato prod)','Acurácia': acc_500},
]).sort_values('Acurácia', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
cores_f = ['#27ae60' if '★' in m else '#3498db' if 'default' in m
           else '#e67e22' if 'XGB' in m else '#e74c3c' if 'balanced' in m
           else '#95a5a6' for m in df_final['Modelo']]
ax.barh(df_final['Modelo'], df_final['Acurácia'], color=cores_f, edgecolor='none')
ax.axvline(0.74, color='red', linestyle='--', linewidth=1.2, label='KNN produção atual (0.74)')
ax.set_xlim(0.6, 1.02)
ax.set_xlabel('Acurácia (holdout 30%)')
ax.set_title('Evolução de Acurácia — Todos os Modelos')
for i, (_, row) in enumerate(df_final.iterrows()):
    ax.text(row['Acurácia'] + 0.003, i, f"{row['Acurácia']:.4f}", va='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()

Teste final: RF com 500 árvores e `max_features=0.4` (sem balanced) → **acc=0.8791**, ganho de apenas **+0.01pp** sobre o default (200 árvores). 34.8 segundos de treino vs 7.7s do default. O retorno decrescente é claro: entre 200 e 500 árvores, o modelo convergiu — mais árvores não ajudam. **Decisão final: RF 200 trees, default parameters** — melhor custo-benefício (acc=0.879, treino=7.7s). A jornada KNN→RF mostrou que a escolha do algoritmo tem impacto muito maior que qualquer tuning de hiperparâmetros. O modelo de produção ficou definido.

## Decisão de modelo — visualização (radar + trade-off)

Gráficos **não-barra** para o slide de decisão. O **radar** mostra que *nenhum* modelo domina todos os critérios: o KNN ponderado lidera em **explicabilidade**, **métricas de similaridade** e **alinhamento ao enunciado** (que pede padrões similares, não classificação); o RandomForest lidera em **acurácia**. Isso justifica o **híbrido: KNN no comando (busca por similaridade) + RF como confirmação**.

> Acurácia e tempo de treino são **medidos** (células acima). Explicabilidade, similaridade e alinhamento são **notas qualitativas (0–1)** — ajuste se quiser.

In [ ]:
# ── Decisão de modelo (não-barra): KNN vs RF vs XGBoost ───────────────────
import numpy as np
import matplotlib.pyplot as plt

# paleta do deck (azul/verde) — gera PNGs idênticos aos do slide 9
BG, TXT, MUTED = "#010D30", "#FFFFFF", "#A8B3C7"
C_KNN, C_RF, C_XGB, NEON = "#10F5A3", "#5B9BD5", "#C7923E", "#10F5A3"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TXT, "axes.labelcolor": TXT, "xtick.color": MUTED, "ytick.color": MUTED,
})

# 1) RADAR multicritério
criterios = ["Acurácia", "Explicabilidade", "Métricas de\nsimilaridade",
             "Alinhamento\nao enunciado", "Leveza\n(custo op.)"]
modelos = {
    "KNN ponderado (50)": ([0.74, 0.95, 1.00, 1.00, 0.85], C_KNN),
    "RandomForest 200":   ([0.879, 0.45, 0.10, 0.40, 0.80], C_RF),
    "XGBoost 300":        ([0.840, 0.50, 0.10, 0.40, 0.40], C_XGB),
}
N = len(criterios)
ang = np.linspace(0, 2*np.pi, N, endpoint=False).tolist(); ang += ang[:1]
fig = plt.figure(figsize=(8, 6.5)); ax = plt.subplot(111, polar=True)
ax.set_facecolor(BG); ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1); ax.set_ylim(0, 1)
ax.set_xticks(ang[:-1]); ax.set_xticklabels(criterios, color=TXT)
ax.set_rgrids([0.25, 0.5, 0.75, 1.0], labels=["", "0.5", "", "1.0"], color=MUTED)
ax.grid(color="#2A3F66"); ax.spines["polar"].set_color(MUTED)
for nome, (vals, cor) in modelos.items():
    v = vals + vals[:1]
    lw = 3.2 if nome.startswith("KNN") else 1.7
    a = 0.27 if nome.startswith("KNN") else 0.08
    ax.plot(ang, v, color=cor, linewidth=lw, label=nome); ax.fill(ang, v, color=cor, alpha=a)
ax.set_title("Decisão de modelo — nenhum domina todos os critérios",
             color=TXT, fontweight="bold", pad=26)
ax.legend(loc="upper left", bbox_to_anchor=(-0.28, 1.12), frameon=False, fontsize=9, labelcolor=TXT)
plt.tight_layout(); plt.savefig("decisao_modelos_radar.png", dpi=150, bbox_inches="tight"); plt.show()

# 2) SCATTER trade-off (acurácia × tempo de treino; bolha = explicabilidade)
pts = {"KNN ponderado (50)": (0.1, 0.74, 0.95, C_KNN),
       "RandomForest 200": (7.7, 0.879, 0.45, C_RF),
       "XGBoost 300": (23.3, 0.840, 0.50, C_XGB)}
fig2, ax2 = plt.subplots(figsize=(8, 5.5)); ax2.set_facecolor(BG)
for nome, (t, acc, expl, cor) in pts.items():
    ax2.scatter(t, acc, s=600*expl + 200, color=cor, alpha=0.85,
                edgecolors=TXT, linewidths=1.4, zorder=3)
    ax2.annotate(f"{nome}\nacc={acc:.3f}", (t, acc), xytext=(t+0.6, acc+0.008),
                 color=TXT, fontsize=9, fontweight="bold" if nome.startswith("KNN") else "normal")
ax2.axhline(0.74, color=NEON, ls="--", lw=1, alpha=0.7)
ax2.set_xlabel("Tempo de treino (s)  →  menor é melhor"); ax2.set_ylabel("Acurácia (holdout 30%)")
ax2.set_title("Trade-off treino × acurácia  (bolha = explicabilidade)", color=TXT, fontweight="bold")
ax2.set_xlim(-1.5, 27); ax2.set_ylim(0.70, 0.92)
for s in ax2.spines.values(): s.set_color(MUTED)
ax2.grid(color="#2A3F66", alpha=0.5)
plt.tight_layout(); plt.savefig("decisao_modelos_scatter.png", dpi=150, bbox_inches="tight"); plt.show()


### Justificativa do modelo de produção — RF 200 default

Gráfico de **ganho marginal × custo**: cada alternativa comparada ao RF 200 default (baseline = 0). Mostra que **nenhuma compensa**: o tuning com `class_weight='balanced'` *piorou* (−0.42pp); 500 árvores rendem só **+0.01pp** por ~2,5× o custo; o XGBoost é 3× mais lento por acurácia menor. → **RF 200 default vence em custo-benefício.**

In [ ]:
# ── Justificativa: por que RF 200 default (ganho marginal vs custo) ────────
import matplotlib.pyplot as plt
BG, TXT, MUTED = "#010D30", "#FFFFFF", "#A8B3C7"
NEON, BLUE, AMBER, GREY = "#10F5A3", "#5B9BD5", "#E0A93B", "#7A8AA8"
plt.rcParams.update({"figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TXT, "axes.labelcolor": TXT, "xtick.color": MUTED, "ytick.color": MUTED})

BASE = 0.8790  # RF 200 default (acurácia holdout)
linhas = [
    ("KNN ponderado (50)", 0.740, "instantâneo, mas é p/ recuperação", GREY),
    ("XGBoost 300",        0.840, "23.3s — 3× mais lento", GREY),
    ("RF tuned (balanced)",0.8748,"+ complexidade -> PIOROU", AMBER),
    ("RF 500 (max_feat .4)",0.8791,"~2,5x custo -> +0.01pp", BLUE),
    ("RF 200 default  *",  0.8790,"7.7s — escolhido", NEON),
]
linhas.sort(key=lambda r: r[1])
labels = [r[0] for r in linhas]; deltas = [(r[1]-BASE)*100 for r in linhas]
notas = [r[2] for r in linhas]; cores = [r[3] for r in linhas]; y = range(len(linhas))

fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.axvline(0, color=NEON, lw=1.6, alpha=0.8)
ax.text(0.05, len(linhas)-0.35, "baseline = RF 200 default", color=NEON, fontsize=8.5)
for yi, d, c, lab in zip(y, deltas, cores, labels):
    ax.plot([0, d], [yi, yi], color=c, lw=2.5, zorder=2, alpha=0.9)
    big = "*" in lab
    ax.scatter(d, yi, s=320 if big else 170, color=c, zorder=3, edgecolors=TXT,
               linewidths=1.4 if big else 0.8)
for yi, d, n in zip(y, deltas, notas):
    off = 0.25 if d >= 0 else -0.25; ha = "left" if d >= 0 else "right"
    ax.annotate(f"{d:+.2f} pp", (d, yi), xytext=(d+off, yi+0.18), ha=ha, color=TXT,
                fontsize=9.5, fontweight="bold")
    ax.annotate(n, (d, yi), xytext=(d+off, yi-0.22), ha=ha, color=MUTED, fontsize=8)
ax.set_yticks(list(y)); ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("Diferença de acurácia vs RF 200 default  (pontos percentuais)")
ax.set_title("Por que o RF 200 default — ganho marginal não paga o custo",
             color=TXT, fontweight="bold", fontsize=14, pad=14)
ax.set_xlim(-15.5, 3.5); ax.set_ylim(-0.6, len(linhas)-0.2)
for s in ax.spines.values(): s.set_color(MUTED)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.grid(axis="x", color="#2A3F66", alpha=0.5)
plt.tight_layout(); plt.savefig("justificativa_rf_default.png", dpi=150, bbox_inches="tight"); plt.show()
